# Exercise 7: From the log to the lake

Every run of your job does three things: **read** the new records of the topic `logs`,
**write** them into the lake (Parquet files in `data/lake/`), **remember** where it stopped.

This notebook gives you the building blocks (`tools/lakelib.py`). You write one function,
`run_once()`. Save your final version of this notebook into your `solution/` folder.

In [1]:
import sys; sys.path.append("/home/jovyan/tools")
from lakelib import *

## 1. The example: a job that copies everything

It reads the topic from the beginning, every time. Run the next two cells and look at the
checker's last line.

In [2]:
def run_once():
    new = read_new(None)          # None = from the beginning of the topic
    write_files(new, LAKE)        # straight into the lake

reset()                           # empty lake
run_once()
check()

reset: lake, staging and state deleted
read_new: 2,900,000 new records
write_files: 2,900,000 records -> /home/jovyan/data/lake/logs/run-20261123-191516-8c8676

Lake check — topic 'logs' (4 partitions)   2026-11-23 19:15:42
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 889,738)     889,738           0          0           0
        1           [0, 730,192)     730,192           0          0           0
        2           [0, 590,536)     590,536           0          0           0
        3           [0, 689,534)     689,534           0          0           0
-------------------------------------------------------------------------------
    total      2,900,000 records   2,900,000           0          0           0

files under /home/jovyan/data/lake: 4 data files, 126.8 MB, 31.70 MB pe

{'topic': 2900000,
 'lake': 2900000,
 'dup': 0,
 'miss': 0,
 'lag': 0,
 'files': 4,
 'mb': 126.793331,
 'ok': True,
 'problems': []}

In [3]:
run_once()                        # the next run
check()

read_new: 2,900,000 new records
write_files: 2,900,000 records -> /home/jovyan/data/lake/logs/run-20261123-191542-a2fa66

Lake check — topic 'logs' (4 partitions)   2026-11-23 19:15:58
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 889,738)   1,779,476     889,738          0           0
        1           [0, 730,192)   1,460,384     730,192          0           0
        2           [0, 590,536)   1,181,072     590,536          0           0
        3           [0, 689,534)   1,379,068     689,534          0           0
-------------------------------------------------------------------------------
    total      2,900,000 records   5,800,000   2,900,000          0           0

files under /home/jovyan/data/lake: 8 data files, 253.6 MB, 31.70 MB per file on average
check took 0.1 s

Exa

{'topic': 2900000,
 'lake': 5800000,
 'dup': 2900000,
 'miss': 0,
 'lag': 0,
 'files': 8,
 'mb': 253.586662,
 'ok': False,
 'problems': ['2,900,000 duplicates']}

## 2. The building blocks

| Block | What it does |
|---|---|
| `load_state()` | where the last run stopped, from a small state file (`None` if there is none) |
| `save_state(new.end)` | stores where this run stopped in the state file |
| `lake_position()` | where the last run stopped, read from the lake itself (`None` if the lake is empty) |
| `read_new(start)` | the records after `start`, up to the end of the topic *now* (`new.end`) |
| `write_files(new, folder)` | writes them as Parquet into a new run folder inside `LAKE` or `STAGING` |
| `publish(run)` | moves a finished run folder from `STAGING` into the lake: one rename, a single step |

`LAKE` is what readers see; `STAGING` is a folder next to it that nobody reads.
`new.df.show()` shows the records of a run as a Spark DataFrame.

## 3. Your architecture

Write `run_once()` for the architecture you chose (A, B or C, see the README).
The order of the calls *is* your architecture.

In [14]:
def run_once():
    # 1. where did the last run stop?
    start = load_state()
    # 2. read what is new
    new = read_new(start)
    # 3. write it
    publish(write_files(new, STAGING))
    # 4. remember where you stopped (if your architecture needs it)
    save_state(new.end)

## 4. Check it

`check_all` empties the lake, then runs your job in five situations: a first run, new records,
nothing new, a run that stops **while** writing, a run that stops **after** writing.
It takes 2-4 minutes.

Architecture C: readers use a view, so check through it: `check_all(run_once, lake_sql=DEDUP_SQL)`.

In [15]:
check_all(run_once)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             4,400,000           0        0           0  24.2s  PASS
2 new records arrive, run again         4,500,000           0        0           0   3.1s  PASS
3 nothing new, run again                4,500,000           0        0           0   0.1s  PASS
4 run stops WHILE writing: right after  4,500,000           0        0     100,000         PASS
  ... then the next run                 4,600,000           0        0           0   3.0s  PASS
5 run stops AFTER writing: right after  4,600,000           0        0     100,000         PASS
  ... then the next run                 4,700,000           0        0           0   3.0s  PASS
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 4400000,
   'lake': 4400000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 152.545454,
   'ok': True,
   'problems': []},
  24.21947193145752),
 ('2 new records arrive, run again',
  {'topic': 4500000,
   'lake': 4500000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 153.596938,
   'ok': True,
   'problems': []},
  3.060877561569214),
 ('3 nothing new, run again',
  {'topic': 4500000,
   'lake': 4500000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 153.596938,
   'ok': True,
   'problems': []},
  0.05519843101501465),
 ('4 run stops WHILE writing: right after',
  {'topic': 4600000,
   'lake': 4500000,
   'dup': 0,
   'miss': 0,
   'lag': 100000,
   'files': 8,
   'mb': 153.596938,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 4600000,
   'lake': 4600000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 12,
   'mb': 154.703437,
   'ok': True,
   'problems': []},


In [11]:
### lool 
def run_once():
    # 1. where did the last run stop?
    start = load_state()
    # 2. read what is new
    new = read_new(start)
    # 3. write it
    save_state(new.end)
check_all(run_once)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                                     0           0        0   3,800,000   0.0s  PASS
2 new records arrive, run again                 0           0        0   3,900,000   0.1s  PASS
3 nothing new, run again                        0           0        0   3,900,000   0.0s  PASS
4 run stops WHILE writing (did not stop!)
  ... then the next run                         0           0        0   4,000,000   0.0s  PASS
5 run stops AFTER writing (did not stop!)
  ... then the next run                         0           0        0   4,100,000   0.0s  PASS
-----------------------------------------------------------------------------------------------
Exactly once after every run: NO — see the rows marked FAIL


[('1 first run',
  {'topic': 3800000,
   'lake': 0,
   'dup': 0,
   'miss': 0,
   'lag': 3800000,
   'ok': True,
   'problems': []},
  0.0031125545501708984),
 ('2 new records arrive, run again',
  {'topic': 3900000,
   'lake': 0,
   'dup': 0,
   'miss': 0,
   'lag': 3900000,
   'ok': True,
   'problems': []},
  0.05434584617614746),
 ('3 nothing new, run again',
  {'topic': 3900000,
   'lake': 0,
   'dup': 0,
   'miss': 0,
   'lag': 3900000,
   'ok': True,
   'problems': []},
  0.0031554698944091797),
 ('4 run stops WHILE writing (did not stop!)', None, None),
 ('  ... then the next run',
  {'topic': 4000000,
   'lake': 0,
   'dup': 0,
   'miss': 0,
   'lag': 4000000,
   'ok': True,
   'problems': []},
  0.0026412010192871094),
 ('5 run stops AFTER writing (did not stop!)', None, None),
 ('  ... then the next run',
  {'topic': 4100000,
   'lake': 0,
   'dup': 0,
   'miss': 0,
   'lag': 4100000,
   'ok': True,
   'problems': []},
  0.002798318862915039)]